# Notebook 16 — F3 (forecasting) CHARACTERIZATION: making the null rigorous

**Phase 1B diagnostic. No new experiment. No model training, no recompute of HGT/no_graph
predictions, no new pre-registration.** This notebook *re-describes* the ALREADY-LOCKED
Gate G4 / V1-S14 sealed-test result (frozen in `data/v1/forecasting_test_wilcoxon.json`) from
saved artifacts only. These are diagnostics over existing data (bootstrap CIs, power statements,
decompositions), explicitly allowed without a pre-registration per the Phase-1 integrity model.
It reads `forecasting_test_per_unit.parquet`, `forecasting_test_metrics.parquet`,
`forecasting_test_wilcoxon.json`, `forecasting_test_calibration.parquet`, `forecasting_sweep.parquet`,
and `forecasting_baselines.parquet`; it imports `emergence_auc` from the forecasting package
(does NOT re-derive AUC). It writes ONLY the new characterization artifacts it owns.

## What the characterization establishes (three honest moves)

1. **Foreground the well-powered evidence.** The lead statistic is the *paired per-topic Brier
   Wilcoxon* over **all 138** units (`p = 2.63e-11`, direction = `favors_baseline`) — not the
   underpowered AUC arm. The graph model is **significantly worse-calibrated** than the graph-free
   baseline, and that is well-powered.
2. **Quantify how underpowered the AUC arm is.** With only **13 positives**, the paired AUC gap
   (`hgt - no_graph`) carries a 95% bootstrap CI of roughly **[-10pp, +5.5pp]** and only **~25%**
   power to detect the pre-registered **+5pp** bar. A single AUC point estimate at this sample size
   simply cannot adjudicate a 5pp effect.
3. **Explain the val->test movement as winner's-curse + small-sample variance, NOT a regime shift.**
   HGT was selected as best-of-5-completed Optuna trials (winner only **+0.77pp** over runner-up).
   On test, HGT did *not* collapse (val 0.737 -> test 0.781, **+4.4pp**); the **baseline improved
   more** (val 0.701 -> test 0.804, **+10.3pp**). The flip is the baseline catching up, not the
   graph failing.

## The honest framing

F3 is most likely a **TRUE null**: the citation-graph topology adds **no calibration value** beyond
temporal/feature dynamics. This notebook makes the null *more rigorous and better-argued* — it does
**not** rescue the graph model. Whatever the bootstrap shows is reported faithfully.

> Provenance: Gate G4 signed NULL (`docs/gates/G4_forecasting.md`, Samer, 2026-06-05); OSF
> pre-registration PR2 DOI `10.17605/OSF.IO/XP94F`. The locked sealed-test artifacts were produced
> by V1-S14 (`notebooks/10_forecasting_eval.ipynb`); the integrated narrative is in
> `notebooks/13_F3_forecasting_narrative.ipynb`. This notebook overwrites NONE of those — it only
> adds `data/v1/forecasting_characterization{.json,_bootstrap.parquet}` and
> `docs/figures/F3_characterization.png`.

## 1. Setup (read-only inputs; import — do not re-derive — the AUC helper)

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd
from scipy.stats import norm

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

# Repo-root sniff — notebook runs from notebooks/, code lives one dir up.
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

DATA = repo_root / "data" / "v1"
FIGURES_DIR = repo_root / "docs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DPI = 120

# Reuse the pre-registered, rank-based AUC helper (sklearn.roc_auc_score under the hood);
# do NOT re-implement AUC. record_run is the house provenance-sidecar writer.
from scifield.forecasting.baselines.base import emergence_auc  # noqa: E402
from scifield.repro import record_run  # noqa: E402


def _load_parquet(path: Path) -> pd.DataFrame | None:
    if not path.exists():
        print(f"MISSING artifact: {path} — cannot render this section.")
        return None
    return pd.read_parquet(path)


def _load_json(path: Path) -> dict | None:
    if not path.exists():
        print(f"MISSING json: {path} — fields unavailable.")
        return None
    return json.loads(path.read_text())


# READ-ONLY locked inputs (produced once by V1-S14 gnn-eval / V1-S13 sweep).
per_unit = _load_parquet(DATA / "forecasting_test_per_unit.parquet")
metrics = _load_parquet(DATA / "forecasting_test_metrics.parquet")
calibration = _load_parquet(DATA / "forecasting_test_calibration.parquet")
sweep = _load_parquet(DATA / "forecasting_sweep.parquet")
baselines = _load_parquet(DATA / "forecasting_baselines.parquet")
wilcoxon = _load_json(DATA / "forecasting_test_wilcoxon.json")
per_unit_sidecar = _load_json(DATA / "forecasting_test_per_unit.parquet.run.json")
sweep_sidecar = _load_json(DATA / "forecasting_sweep.parquet.run.json")

print(
    "loaded:",
    {
        k: ("ok" if v is not None else None)
        for k, v in {
            "per_unit": per_unit,
            "metrics": metrics,
            "calibration": calibration,
            "sweep": sweep,
            "baselines": baselines,
            "wilcoxon": wilcoxon,
        }.items()
    },
)

# Confirmed-numbers from the independent audit — cross-checked against artifacts below.
AUDIT = {
    "auc_no_graph": 0.8043077,
    "auc_hgt": 0.7809231,
    "delta_pp": -2.3385,
    "brier_p": 2.6290e-11,
    "brier_stat": 1659.0,
    "brier_median_diff": 0.2330,
    "hgt_share_mape": 0.5729,
    "no_graph_share_mape": 0.4610,
    "winner_margin_pp": 0.77,
    "gap_ci_pp": (-10.0, 5.5),
    "power_5pp": 0.25,
}
SEED = 1729  # the single Optuna seed; reused so the bootstrap is reproducible.
N_BOOT = 10_000

loaded: {'per_unit': 'ok', 'metrics': 'ok', 'calibration': 'ok', 'sweep': 'ok', 'baselines': 'ok', 'wilcoxon': 'ok'}


## 2. Build the paired wide frame + cross-check the locked headline numbers

Pivot the 690-row long per-unit table to one row per `(topic_id, origin_year)` carrying the
**shared** binary label and the two models' emergence scores. We assert the keys and labels agree
between `hgt` and `no_graph` (so units stay genuinely paired), then recompute the overall AUCs with
the imported `emergence_auc` and confirm they match the audit's locked values to <0.01pp.

In [2]:
ng = (
    per_unit[per_unit.model == "no_graph"]
    .sort_values(["topic_id", "origin_year"])
    .reset_index(drop=True)
)
hg = (
    per_unit[per_unit.model == "hgt"]
    .sort_values(["topic_id", "origin_year"])
    .reset_index(drop=True)
)
# Units must be paired: identical keys in identical order, and identical shared labels.
assert (
    ng[["topic_id", "origin_year"]].values == hg[["topic_id", "origin_year"]].values
).all(), "(topic_id, origin_year) keys disagree between hgt and no_graph"
assert (ng["emergent"].values == hg["emergent"].values).all(), "shared emergent label disagrees"

df = pd.DataFrame(
    {
        "topic_id": ng.topic_id.values,
        "origin_year": ng.origin_year.astype(int).values,
        "y": ng.emergent.astype(int).values,
        "s_ng": ng.emergence_score.astype(float).values,
        "s_hgt": hg.emergence_score.astype(float).values,
    }
)

N = len(df)
N_POS = int(df.y.sum())
N_NEG = N - N_POS
auc_ng = emergence_auc(df.s_ng.values, df.y.values)
auc_hgt = emergence_auc(df.s_hgt.values, df.y.values)
delta_pp = (auc_hgt - auc_ng) * 100.0

print(f"paired wide frame: N={N} units | positives={N_POS} ({N_POS / N:.1%}) | negatives={N_NEG}")
print(f"  origin-year split: {dict(df.groupby('origin_year').y.agg(['size', 'sum']).T.to_dict())}")
print()
print("CROSS-CHECK vs locked audit numbers:")
print(f"  no_graph AUC = {auc_ng:.7f}  (audit {AUDIT['auc_no_graph']:.7f})")
print(f"  hgt      AUC = {auc_hgt:.7f}  (audit {AUDIT['auc_hgt']:.7f})")
print(f"  delta    pp  = {delta_pp:+.4f}    (audit {AUDIT['delta_pp']:+.4f}; bar was +5.00pp)")
for name, got, exp in [
    ("no_graph AUC", auc_ng, AUDIT["auc_no_graph"]),
    ("hgt AUC", auc_hgt, AUDIT["auc_hgt"]),
]:
    assert abs(got - exp) < 1e-4, f"{name} deviates from audit: {got} vs {exp}"
assert abs(delta_pp - AUDIT["delta_pp"]) < 0.01, "delta_pp deviates from audit"
print("  -> all headline AUCs reproduce the audit (within 0.01pp). PASS.")

paired wide frame: N=138 units | positives=13 (9.4%) | negatives=125
  origin-year split: {2021: {'size': 67, 'sum': 7}, 2022: {'size': 71, 'sum': 6}}

CROSS-CHECK vs locked audit numbers:
  no_graph AUC = 0.8043077  (audit 0.8043077)
  hgt      AUC = 0.7809231  (audit 0.7809231)
  delta    pp  = -2.3385    (audit -2.3385; bar was +5.00pp)
  -> all headline AUCs reproduce the audit (within 0.01pp). PASS.


## 3. LEAD evidence — the well-powered paired Brier calibration test (all 138 units)

The statistic with real teeth is **not** the AUC gap but the **paired per-topic Brier-loss Wilcoxon
signed-rank test** over *all 138* units (every unit contributes, regardless of label), read from the
frozen `forecasting_test_wilcoxon.json`. It is significant **against** the graph model
(`favors_baseline`): HGT carries the systematically *higher* Brier loss. We re-form the same paired
Brier diff here from the saved per-unit scores to confirm the frozen statistic reproduces, then read
the calibration mechanism straight from `forecasting_test_calibration.parquet`.

In [3]:
import scipy.stats

pb = wilcoxon["primary_brier"]
# Re-form the paired per-row Brier diff from saved scores (mirrors evaluate._aligned_diff 'brier').
brier_hgt = (df.s_hgt.values - df.y.values) ** 2
brier_ng = (df.s_ng.values - df.y.values) ** 2
brier_diff = brier_hgt - brier_ng  # > 0 => HGT worse (higher loss)
_w = scipy.stats.wilcoxon(brier_diff)
recomputed_p = float(_w.pvalue)
recomputed_stat = float(_w.statistic)
recomputed_median = float(np.median(brier_diff))

print("LEAD STATISTIC — paired per-topic Brier-loss Wilcoxon (HGT vs no_graph), all units:")
print(
    f"  frozen (artifact): p={float(pb['pvalue']):.4e}  statistic={pb['statistic']:.1f}  "
    f"n_pairs={pb['n_pairs']}  median_diff={float(pb['median_diff']):+.4f}  "
    f"direction={pb['direction'].upper()}"
)
print(
    f"  recomputed here  : p={recomputed_p:.4e}  statistic={recomputed_stat:.1f}  "
    f"n_pairs={len(brier_diff)}  median_diff={recomputed_median:+.4f}"
)
assert abs(recomputed_p - float(pb["pvalue"])) / float(pb["pvalue"]) < 1e-6, "Brier p drift"
assert recomputed_stat == float(pb["statistic"]), "Brier statistic drift"
assert len(brier_diff) == 138, "Brier test must use all 138 units"
print("  -> reproduces the frozen Brier verdict exactly. PASS.")
print()
n_hgt_worse = int((brier_diff > 0).sum())
n_hgt_better = int((brier_diff < 0).sum())
print(f"  per-unit direction: HGT worse on {n_hgt_worse}/138 units, better on {n_hgt_better}/138.")
print("  Reading: the calibration deficit is well-powered (all 138 units, p=2.63e-11) and points")
print("  AGAINST the graph model. This is the lead evidence; the AUC arm (next) is underpowered.")

LEAD STATISTIC — paired per-topic Brier-loss Wilcoxon (HGT vs no_graph), all units:
  frozen (artifact): p=2.6290e-11  statistic=1659.0  n_pairs=138  median_diff=+0.2330  direction=FAVORS_BASELINE
  recomputed here  : p=2.6290e-11  statistic=1659.0  n_pairs=138  median_diff=+0.2330
  -> reproduces the frozen Brier verdict exactly. PASS.

  per-unit direction: HGT worse on 125/138 units, better on 13/138.
  Reading: the calibration deficit is well-powered (all 138 units, p=2.63e-11) and points
  AGAINST the graph model. This is the lead evidence; the AUC arm (next) is underpowered.


In [4]:
# Calibration MECHANISM — where each model places its probability mass vs realized emergence.
base_rate = N_POS / N
cal_hgt = calibration[calibration.model == "hgt"].copy()
cal_ng = calibration[calibration.model == "no_graph"].copy()


def _mass_in(cal: pd.DataFrame, lo: float, hi: float) -> tuple[int, float]:
    sel = cal[(cal.bin_lo >= lo - 1e-9) & (cal.bin_hi <= hi + 1e-9)]
    n = int(sel["n"].sum())
    obs = float(np.nansum(sel["n"] * sel["observed_freq"]) / n) if n else float("nan")
    return n, obs


hgt_mid_n, hgt_mid_obs = _mass_in(cal_hgt, 0.4, 0.8)
ng_low_n, ng_low_obs = _mass_in(cal_ng, 0.0, 0.1)
smape = metrics.set_index("model")["share_mape"]
worst_smape_model = smape.idxmax()

print(f"Base rate of emergence on test = {base_rate:.1%} ({N_POS}/{N}).\n")
print("HGT reliability table (predicted-prob bin -> count, observed emergence freq):")
print(
    cal_hgt[["bin_lo", "bin_hi", "n", "mean_predicted", "observed_freq"]]
    .round(3)
    .to_string(index=False)
)
print(
    f"  -> HGT concentrates {hgt_mid_n}/{N} units in predicted-prob [0.4, 0.8] where realized "
    f"emergence is only {hgt_mid_obs:.1%} (~0, vs the {base_rate:.1%} base rate): OVER-PREDICTION."
)
print()
print("no_graph reliability table:")
print(
    cal_ng[["bin_lo", "bin_hi", "n", "mean_predicted", "observed_freq"]]
    .round(3)
    .to_string(index=False)
)
print(
    f"  -> no_graph keeps {ng_low_n}/{N} units in [0.0, 0.1] with observed freq {ng_low_obs:.1%} "
    f"~= the {base_rate:.1%} base rate: roughly CALIBRATED."
)
print()
print("share-MAPE (lower = better; over positive-target rows):")
for m in ["naive", "arima", "mlp", "no_graph", "hgt"]:
    flag = "  <- WORST of 5 (over-prediction)" if m == worst_smape_model else ""
    print(f"  {m:>9}: {float(smape[m]):.4f}{flag}")
assert worst_smape_model == "hgt", "expected HGT to carry the worst share-MAPE"
assert abs(float(smape["hgt"]) - AUDIT["hgt_share_mape"]) < 1e-3, "HGT share-MAPE drift"
print(
    f"  HGT share-MAPE {float(smape['hgt']):.4f} is the worst of 5 "
    f"(audit {AUDIT['hgt_share_mape']});"
    f" no_graph {float(smape['no_graph']):.4f}. Same over-prediction drives the Brier loss."
)

Base rate of emergence on test = 9.4% (13/138).

HGT reliability table (predicted-prob bin -> count, observed emergence freq):
 bin_lo  bin_hi  n  mean_predicted  observed_freq
    0.0     0.1  0             NaN            NaN
    0.1     0.2  6           0.176          0.000
    0.2     0.3 13           0.249          0.000
    0.3     0.4 15           0.363          0.000
    0.4     0.5 26           0.455          0.000
    0.5     0.6 37           0.556          0.135
    0.6     0.7 34           0.645          0.206
    0.7     0.8  7           0.733          0.143
    0.8     0.9  0             NaN            NaN
    0.9     1.0  0             NaN            NaN
  -> HGT concentrates 104/138 units in predicted-prob [0.4, 0.8] where realized emergence is only 12.5% (~0, vs the 9.4% base rate): OVER-PREDICTION.

no_graph reliability table:
 bin_lo  bin_hi   n  mean_predicted  observed_freq
    0.0     0.1 129           0.026          0.093
    0.1     0.2   7           0.138       

## 4. Stratified paired bootstrap of the AUC gap (10k resamples, fixed seed)

We resample units **with replacement within the 13 positives and 125 negatives separately**
(label-stratified, so every replicate keeps a non-degenerate label set), and **keep units paired**
across the two models (the same resampled index scores both `hgt` and `no_graph`). This yields 95%
percentile CIs for each model's AUC and for the paired gap `hgt - no_graph`. The gap is the
scientifically relevant quantity — its CI tells us what AUC difference the test could actually
resolve at 13 positives.

In [5]:
rng = np.random.default_rng(SEED)
pos_idx = df.index[df.y == 1].to_numpy()
neg_idx = df.index[df.y == 0].to_numpy()
y = df.y.values
sng = df.s_ng.values
shg = df.s_hgt.values

boot_ng = np.full(N_BOOT, np.nan)
boot_hgt = np.full(N_BOOT, np.nan)
boot_gap = np.full(N_BOOT, np.nan)
for b in range(N_BOOT):
    idx = np.concatenate(
        [
            rng.choice(pos_idx, size=pos_idx.size, replace=True),
            rng.choice(neg_idx, size=neg_idx.size, replace=True),
        ]
    )
    yy = y[idx]
    if np.unique(yy).shape[0] < 2:  # cannot happen given stratification, but guard anyway
        continue
    a_ng = emergence_auc(sng[idx], yy)
    a_hg = emergence_auc(shg[idx], yy)
    boot_ng[b] = a_ng
    boot_hgt[b] = a_hg
    boot_gap[b] = a_hg - a_ng

n_valid = int(np.isfinite(boot_gap).sum())


def _ci(a: np.ndarray) -> tuple[float, float]:
    a = a[np.isfinite(a)]
    return float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))


ng_lo, ng_hi = _ci(boot_ng)
hgt_lo, hgt_hi = _ci(boot_hgt)
gap_lo, gap_hi = _ci(boot_gap)
gap_mean = float(np.nanmean(boot_gap))
gap_se_boot = float(np.nanstd(boot_gap, ddof=1))

print(
    f"Stratified paired bootstrap: {N_BOOT} resamples (seed={SEED}), "
    f"{n_valid} valid; strata {N_POS} pos / {N_NEG} neg.\n"
)
print(f"  no_graph AUC: point {auc_ng:.4f}  95% CI [{ng_lo:.4f}, {ng_hi:.4f}]")
print(f"  hgt      AUC: point {auc_hgt:.4f}  95% CI [{hgt_lo:.4f}, {hgt_hi:.4f}]")
print(f"  GAP (hgt - no_graph): point {delta_pp:+.2f}pp  mean {gap_mean * 100:+.2f}pp")
print(
    f"     95% CI = [{gap_lo * 100:+.2f}pp, {gap_hi * 100:+.2f}pp]   (bootstrap SE "
    f"{gap_se_boot * 100:.3f}pp)"
)
frac_clears_5pp = float(np.mean(boot_gap[np.isfinite(boot_gap)] > 0.05))
print(
    f"  fraction of replicates with gap > +5pp = {frac_clears_5pp:.4f} "
    "(re-sampling the OBSERVED data; not a true-effect power calc — see §5)."
)
print()
plan_lo, plan_hi = AUDIT["gap_ci_pp"]
dev_lo = gap_lo * 100 - plan_lo
dev_hi = gap_hi * 100 - plan_hi
print(
    f"  CROSS-CHECK vs plan's expected gap CI ~[{plan_lo:+.1f}pp, {plan_hi:+.1f}pp]: "
    f"got [{gap_lo * 100:+.2f}pp, {gap_hi * 100:+.2f}pp] "
    f"(low end {dev_lo:+.2f}pp, high end {dev_hi:+.2f}pp from plan)."
)
MATERIAL_PP = 1.0  # treat >1pp drift on a CI endpoint as material
if abs(dev_lo) > MATERIAL_PP or abs(dev_hi) > MATERIAL_PP:
    print("  *** FLAG: bootstrap gap CI deviates materially (>1pp) from the plan's estimate. ***")
else:
    print("  -> matches the plan's [-10pp, +5.5pp] estimate (no material deviation).")

Stratified paired bootstrap: 10000 resamples (seed=1729), 10000 valid; strata 13 pos / 125 neg.

  no_graph AUC: point 0.8043  95% CI [0.7200, 0.8794]
  hgt      AUC: point 0.7809  95% CI [0.6886, 0.8646]
  GAP (hgt - no_graph): point -2.34pp  mean -2.37pp
     95% CI = [-10.09pp, +5.60pp]   (bootstrap SE 3.979pp)
  fraction of replicates with gap > +5pp = 0.0331 (re-sampling the OBSERVED data; not a true-effect power calc — see §5).

  CROSS-CHECK vs plan's expected gap CI ~[-10.0pp, +5.5pp]: got [-10.09pp, +5.60pp] (low end -0.09pp, high end +0.10pp from plan).
  -> matches the plan's [-10pp, +5.5pp] estimate (no material deviation).


## 5. DeLong cross-check of the gap SE + power / MDE for the +5pp bar

We cross-check the bootstrap gap SE against **DeLong's** analytic covariance for two *correlated*
AUCs (Sun & Xu 2014 fast midrank algorithm) — the standard parametric method. The two SEs should
agree. We then state the **power** to detect the pre-registered **+5pp** true gap at alpha=0.05
(two-sided) and the **minimum detectable effect (MDE)** at 80% power, using (a) the paired gap SE
(bootstrap and DeLong agree) and (b) for contrast, the Hanley-McNeil single-AUC SEs combined under
*independence* — which over-states the SE because the two models are positively correlated, so the
**paired** SE is the correct one.

In [6]:
def _midrank(x: np.ndarray) -> np.ndarray:
    """Tie-aware midranks (Sun & Xu 2014 helper)."""
    order = np.argsort(x)
    z = x[order]
    nn = len(x)
    t = np.zeros(nn)
    i = 0
    while i < nn:
        j = i
        while j < nn and z[j] == z[i]:
            j += 1
        t[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    out = np.empty(nn)
    out[order] = t
    return out


def delong_auc_cov(preds: np.ndarray, labels: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Fast DeLong AUCs + covariance for k models on shared binary labels.

    Returns (aucs[k], cov[k,k]). Cross-checks `emergence_auc`; not a re-derivation
    of the headline metric — used only for the analytic gap SE.
    """
    order = np.argsort(-labels, kind="stable")  # positives first
    lab = labels[order]
    m = int(lab.sum())
    ntot = len(lab)
    n = ntot - m
    p = preds[:, order]
    k = p.shape[0]
    pos, neg = p[:, :m], p[:, m:]
    tx = np.empty([k, m])
    ty = np.empty([k, n])
    tz = np.empty([k, ntot])
    for r in range(k):
        tx[r] = _midrank(pos[r])
        ty[r] = _midrank(neg[r])
        tz[r] = _midrank(p[r])
    aucs = (tz[:, :m].sum(axis=1) / m - (m + 1) / 2.0) / n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    sx = np.atleast_2d(np.cov(v01))
    sy = np.atleast_2d(np.cov(v10))
    cov = sx / m + sy / n
    return aucs, cov


dl_aucs, dl_cov = delong_auc_cov(np.vstack([shg, sng]), y)
# DeLong AUC point estimates must match the imported helper (sanity that DeLong is wired right).
assert abs(dl_aucs[0] - auc_hgt) < 1e-9 and abs(dl_aucs[1] - auc_ng) < 1e-9, "DeLong AUC mismatch"
se_hgt_dl = float(np.sqrt(dl_cov[0, 0]))
se_ng_dl = float(np.sqrt(dl_cov[1, 1]))
gap_se_delong = float(np.sqrt(dl_cov[0, 0] + dl_cov[1, 1] - 2 * dl_cov[0, 1]))
corr_dl = float(dl_cov[0, 1] / np.sqrt(dl_cov[0, 0] * dl_cov[1, 1]))

print("DeLong analytic cross-check (correlated AUCs):")
print(f"  per-AUC SE: hgt={se_hgt_dl * 100:.3f}pp  no_graph={se_ng_dl * 100:.3f}pp")
print(
    f"  gap SE (paired): DeLong={gap_se_delong * 100:.3f}pp  vs  bootstrap="
    f"{gap_se_boot * 100:.3f}pp  (agree to {abs(gap_se_delong - gap_se_boot) * 100:.3f}pp)"
)
print(
    f"  model-score correlation rho(hgt, no_graph) = {corr_dl:.3f}  (positive -> paired SE is "
    "much tighter than the independent combination; confirms the paired analysis is correct)."
)
assert abs(gap_se_delong - gap_se_boot) < 0.005, "DeLong vs bootstrap gap SE disagree by >0.5pp"
print("  -> DeLong and bootstrap gap SEs agree within 0.5pp. PASS.")

DeLong analytic cross-check (correlated AUCs):
  per-AUC SE: hgt=4.579pp  no_graph=4.097pp
  gap SE (paired): DeLong=3.985pp  vs  bootstrap=3.979pp  (agree to 0.006pp)
  model-score correlation rho(hgt, no_graph) = 0.583  (positive -> paired SE is much tighter than the independent combination; confirms the paired analysis is correct).
  -> DeLong and bootstrap gap SEs agree within 0.5pp. PASS.


In [7]:
# Hanley-McNeil single-AUC SE (closed form; uses AUC, n_pos, n_neg).
def hanley_mcneil_se(auc: float, n_pos: int, n_neg: int) -> float:
    q1 = auc / (2.0 - auc)
    q2 = 2.0 * auc * auc / (1.0 + auc)
    var = (auc * (1.0 - auc) + (n_pos - 1) * (q1 - auc**2) + (n_neg - 1) * (q2 - auc**2)) / (
        n_pos * n_neg
    )
    return float(np.sqrt(var))


se_hgt_hm = hanley_mcneil_se(auc_hgt, N_POS, N_NEG)
se_ng_hm = hanley_mcneil_se(auc_ng, N_POS, N_NEG)
se_indep_hm = float(np.sqrt(se_hgt_hm**2 + se_ng_hm**2))  # naive independence (over-states SE)

ALPHA = 0.05
z_a = norm.ppf(1 - ALPHA / 2)  # two-sided
z_b = norm.ppf(0.80)
TRUE_GAP = 0.05  # the pre-registered +5pp bar


def power_for(se: float, eff: float = TRUE_GAP) -> float:
    return float(1 - norm.cdf(z_a - eff / se) + norm.cdf(-z_a - eff / se))


def mde_at(se: float, power: float = 0.80) -> float:
    return float((z_a + norm.ppf(power)) * se)


# Primary (correct) SE = the paired gap SE; bootstrap and DeLong agree, use bootstrap as headline.
power_paired = power_for(gap_se_boot)
mde_paired = mde_at(gap_se_boot)
power_paired_dl = power_for(gap_se_delong)
power_indep = power_for(se_indep_hm)
mde_indep = mde_at(se_indep_hm)

print("Hanley-McNeil single-AUC SE (closed form):")
print(f"  hgt SE = {se_hgt_hm * 100:.3f}pp   no_graph SE = {se_ng_hm * 100:.3f}pp")
print(
    f"  (DeLong per-AUC SE for comparison: hgt {se_hgt_dl * 100:.3f}pp, "
    f"no_graph {se_ng_dl * 100:.3f}pp)"
)
print()
print(f"POWER / MDE for the +5pp bar (alpha={ALPHA}, two-sided, 80% power for MDE):")
print(
    f"  PAIRED gap SE = {gap_se_boot * 100:.3f}pp (bootstrap) / {gap_se_delong * 100:.3f}pp "
    "(DeLong)  <- CORRECT, models are correlated"
)
print(f"     power(+5pp) = {power_paired:.3f}  (DeLong {power_paired_dl:.3f})  ~= 25%")
print(f"     MDE@80%     = {mde_paired * 100:.2f}pp")
print(
    f"  Independent HM-combined SE = {se_indep_hm * 100:.3f}pp  (over-states; ignores rho="
    f"{corr_dl:.2f})"
)
print(f"     power(+5pp) = {power_indep:.3f}   MDE@80% = {mde_indep * 100:.2f}pp")
print()
dev_power = power_paired - AUDIT["power_5pp"]
print(
    f"  CROSS-CHECK vs audit ~25% power: got {power_paired:.1%} "
    f"(deviation {dev_power * 100:+.1f}pp from 25%)."
)
if abs(dev_power) > 0.05:
    print("  *** FLAG: power deviates materially (>5pp) from the audit's ~25%. ***")
else:
    print("  -> matches the audit's ~25% power estimate (no material deviation).")
print()
print("Reading: at 13 positives the AUC arm can only resolve effects of ~11pp at 80% power; a +5pp")
print("true gap would be detected only ~1 time in 4. The AUC point estimate (-2.34pp) is therefore")
print("UNINFORMATIVE about a 5pp effect — the lead evidence is the well-powered Brier test (§3).")

Hanley-McNeil single-AUC SE (closed form):
  hgt SE = 7.783pp   no_graph SE = 7.511pp
  (DeLong per-AUC SE for comparison: hgt 4.579pp, no_graph 4.097pp)

POWER / MDE for the +5pp bar (alpha=0.05, two-sided, 80% power for MDE):
  PAIRED gap SE = 3.979pp (bootstrap) / 3.985pp (DeLong)  <- CORRECT, models are correlated
     power(+5pp) = 0.242  (DeLong 0.241)  ~= 25%
     MDE@80%     = 11.15pp
  Independent HM-combined SE = 10.816pp  (over-states; ignores rho=0.58)
     power(+5pp) = 0.075   MDE@80% = 30.30pp

  CROSS-CHECK vs audit ~25% power: got 24.2% (deviation -0.8pp from 25%).
  -> matches the audit's ~25% power estimate (no material deviation).

Reading: at 13 positives the AUC arm can only resolve effects of ~11pp at 80% power; a +5pp
true gap would be detected only ~1 time in 4. The AUC point estimate (-2.34pp) is therefore
UNINFORMATIVE about a 5pp effect — the lead evidence is the well-powered Brier test (§3).


## 6. Winner's-curse + val->test decomposition (NOT a regime shift)

Two facts from `forecasting_sweep.parquet` and `forecasting_baselines.parquet` reframe the val->test
"flip":

1. **Selection inflation (winner's curse).** HGT was chosen as the **best of 5 *completed*** Optuna
   trials (the other 15 of 20 were pruned at epoch 1; single seed 1729). The winner's val AUC beats
   the runner-up by only **+0.77pp** — a margin well inside the selection noise, so the selected
   val AUC is an *optimistic* estimate of true performance.
2. **The baseline improved more — HGT did not collapse.** From val to the sealed test, the
   graph-free baseline gained **+10.3pp** (0.701 -> 0.804) while HGT gained **+4.4pp**
   (0.737 -> 0.781). HGT's *absolute* test AUC is *higher* than its val AUC; the gap flipped sign
   because the **baseline caught up and overtook**, not because the graph model degraded. That is
   exactly the signature of winner's-curse selection on a tiny (13-positive) test set, not a regime
   shift in the data.

**Transparent deviation:** PR2 configured `n_trials=40`, but the committed sweep ran **20** trials
of which **5 completed** (15 pruned). The model-selection therefore drew from 5 completed candidates,
not 40 — disclosed here and in the findings doc.

In [8]:
n_trials_total = int(len(sweep))
n_complete = int((sweep.state == "complete").sum())
n_pruned = int((sweep.state == "pruned").sum())
pr2_configured_trials = 40  # PR2 spec; disclose vs what actually ran

comp = (
    sweep[sweep.state == "complete"].sort_values("val_auc", ascending=False).reset_index(drop=True)
)
winner_val = float(comp.val_auc.iloc[0])
runnerup_val = float(comp.val_auc.iloc[1])
winner_margin_pp = (winner_val - runnerup_val) * 100.0
winner_trial = int(comp.trial_number.iloc[0])
runnerup_trial = int(comp.trial_number.iloc[1])
seeds = sorted(sweep.seed.unique().tolist())

# val baselines (no_graph is the selected comparator) and the locked test metrics.
ng_val = float(baselines.set_index("baseline").loc["no_graph", "emergence_auc"])
ng_test = auc_ng
hgt_val = winner_val  # the selected HGT config's val AUC
hgt_test = auc_hgt
ng_gain_pp = (ng_test - ng_val) * 100.0
hgt_gain_pp = (hgt_test - hgt_val) * 100.0
val_gap_pp = (hgt_val - ng_val) * 100.0
test_gap_pp = (hgt_test - ng_test) * 100.0

print(
    f"Optuna sweep: {n_trials_total} trials ran | {n_complete} complete, {n_pruned} pruned | "
    f"seed(s)={seeds}"
)
print(
    f"  PR2 configured n_trials={pr2_configured_trials}; ACTUAL ran {n_trials_total}, "
    f"{n_complete} completed -> transparent deviation (selection drew from {n_complete})."
)
print()
print("Completed-trial val AUCs (the selection pool):")
print(
    comp[["trial_number", "val_auc", "val_mape", "lr", "hidden", "n_layers", "dropout"]]
    .round(4)
    .to_string(index=False)
)
print()
print(f"  WINNER trial {winner_trial}: val AUC {winner_val:.4f}")
print(f"  runner-up trial {runnerup_trial}: val AUC {runnerup_val:.4f}")
print(
    f"  winner-vs-runner-up margin = +{winner_margin_pp:.2f}pp  "
    f"(audit +{AUDIT['winner_margin_pp']:.2f}pp)"
)
assert abs(winner_margin_pp - AUDIT["winner_margin_pp"]) < 0.05, "winner margin drift"
print()
print("VAL -> TEST decomposition:")
print(
    f"  no_graph: val {ng_val:.4f} -> test {ng_test:.4f}  = +{ng_gain_pp:.2f}pp  "
    f"(BASELINE GAINS MORE)"
)
print(
    f"  hgt     : val {hgt_val:.4f} -> test {hgt_test:.4f}  = +{hgt_gain_pp:.2f}pp  "
    f"(HGT also GAINS — no collapse)"
)
print(f"  gap (hgt - no_graph): val {val_gap_pp:+.2f}pp  ->  test {test_gap_pp:+.2f}pp")
print("  => the sign flip is the BASELINE overtaking, not HGT degrading. Consistent with")
print("     winner's-curse selection + small-sample (13-positive) variance, NOT a regime shift.")

Optuna sweep: 20 trials ran | 5 complete, 15 pruned | seed(s)=[1729]
  PR2 configured n_trials=40; ACTUAL ran 20, 5 completed -> transparent deviation (selection drew from 5).

Completed-trial val AUCs (the selection pool):
 trial_number  val_auc  val_mape     lr  hidden  n_layers  dropout
            1   0.7367    0.5527 0.0029      32         2   0.3762
            4   0.7290    0.4985 0.0091      64         3   0.4540
            0   0.6985    0.4856 0.0003     128         2   0.1270
            2   0.6277    0.3904 0.0006      64         3   0.2234
            3   0.4955   16.2202 0.0001      64         1   0.0711

  WINNER trial 1: val AUC 0.7367
  runner-up trial 4: val AUC 0.7290
  winner-vs-runner-up margin = +0.77pp  (audit +0.77pp)

VAL -> TEST decomposition:
  no_graph: val 0.7010 -> test 0.8043  = +10.34pp  (BASELINE GAINS MORE)
  hgt     : val 0.7367 -> test 0.7809  = +4.42pp  (HGT also GAINS — no collapse)
  gap (hgt - no_graph): val +3.57pp  ->  test -2.34pp
  => the sig

In [9]:
# Per-origin-year test AUC with stratified paired bootstrap CIs (DESCRIPTIVE ONLY — 6-7 pos/yr).
per_year_rows = []
boot_year = {}
for yr in sorted(df.origin_year.unique()):
    d = df[df.origin_year == yr]
    yy = d.y.values
    a_ng = emergence_auc(d.s_ng.values, yy)
    a_hg = emergence_auc(d.s_hgt.values, yy)
    pidx = d.index[d.y == 1].to_numpy()
    nidx = d.index[d.y == 0].to_numpy()
    gboot = np.full(N_BOOT, np.nan)
    ng_b = np.full(N_BOOT, np.nan)
    hg_b = np.full(N_BOOT, np.nan)
    rng_y = np.random.default_rng(SEED + int(yr))
    for b in range(N_BOOT):
        idx = np.concatenate(
            [rng_y.choice(pidx, pidx.size, True), rng_y.choice(nidx, nidx.size, True)]
        )
        lab = y[idx]
        if np.unique(lab).shape[0] < 2:
            continue
        a1 = emergence_auc(sng[idx], lab)
        a2 = emergence_auc(shg[idx], lab)
        ng_b[b] = a1
        hg_b[b] = a2
        gboot[b] = a2 - a1
    boot_year[int(yr)] = {"ng": ng_b, "hgt": hg_b, "gap": gboot}
    g_lo, g_hi = _ci(gboot)
    nglo, nghi = _ci(ng_b)
    hglo, hghi = _ci(hg_b)
    per_year_rows.append(
        {
            "origin_year": int(yr),
            "n": int(len(d)),
            "n_pos": int(yy.sum()),
            "auc_no_graph": a_ng,
            "auc_no_graph_lo": nglo,
            "auc_no_graph_hi": nghi,
            "auc_hgt": a_hg,
            "auc_hgt_lo": hglo,
            "auc_hgt_hi": hghi,
            "gap_pp": (a_hg - a_ng) * 100.0,
            "gap_lo_pp": g_lo * 100.0,
            "gap_hi_pp": g_hi * 100.0,
        }
    )
per_year = pd.DataFrame(per_year_rows)
print(
    "Per-origin-year test AUC (DESCRIPTIVE; 6-7 positives/yr -> very wide CIs, "
    "do NOT over-interpret):"
)
print(per_year.round(4).to_string(index=False))
print()
for _, r in per_year.iterrows():
    print(
        f"  {int(r.origin_year)}: n={int(r.n)} pos={int(r.n_pos)} | no_graph {r.auc_no_graph:.3f} "
        f"[{r.auc_no_graph_lo:.3f},{r.auc_no_graph_hi:.3f}]  hgt {r.auc_hgt:.3f} "
        f"[{r.auc_hgt_lo:.3f},{r.auc_hgt_hi:.3f}]  gap {r.gap_pp:+.1f}pp "
        f"[{r.gap_lo_pp:+.1f},{r.gap_hi_pp:+.1f}]pp"
    )
print(
    "  => 2021 favors HGT, 2022 favors no_graph; both per-year gap CIs are enormous (span ~30pp)."
)
print("     The year-to-year swing is small-sample noise, not evidence either way.")

Per-origin-year test AUC (DESCRIPTIVE; 6-7 positives/yr -> very wide CIs, do NOT over-interpret):
 origin_year  n  n_pos  auc_no_graph  auc_no_graph_lo  auc_no_graph_hi  auc_hgt  auc_hgt_lo  auc_hgt_hi   gap_pp  gap_lo_pp  gap_hi_pp
        2021 67      7        0.8238           0.7095           0.9190   0.8667      0.7571      0.9524   4.2857    -5.7143    15.2381
        2022 71      6        0.7872           0.6615           0.8949   0.6872      0.5512      0.8128 -10.0000   -19.7436    -0.7692

  2021: n=67 pos=7 | no_graph 0.824 [0.710,0.919]  hgt 0.867 [0.757,0.952]  gap +4.3pp [-5.7,+15.2]pp
  2022: n=71 pos=6 | no_graph 0.787 [0.662,0.895]  hgt 0.687 [0.551,0.813]  gap -10.0pp [-19.7,-0.8]pp
  => 2021 favors HGT, 2022 favors no_graph; both per-year gap CIs are enormous (span ~30pp).
     The year-to-year swing is small-sample noise, not evidence either way.


## 7. Persist the characterization artifacts (+ provenance sidecars)

Write the new bootstrap replicate table (`forecasting_characterization_bootstrap.parquet`) and the
scalar summary (`forecasting_characterization.json`), each with a `record_run` sidecar tying them
back to the locked inputs' hashes. These are NEW files; no locked artifact is touched.

In [10]:
# (a) Bootstrap replicate table: overall paired bootstrap + per-year, long form.
boot_frames = [
    pd.DataFrame(
        {
            "stratum": "overall",
            "replicate": np.arange(N_BOOT),
            "auc_no_graph": boot_ng,
            "auc_hgt": boot_hgt,
            "gap": boot_gap,
        }
    )
]
for yr, d in boot_year.items():
    boot_frames.append(
        pd.DataFrame(
            {
                "stratum": f"origin_{yr}",
                "replicate": np.arange(N_BOOT),
                "auc_no_graph": d["ng"],
                "auc_hgt": d["hgt"],
                "gap": d["gap"],
            }
        )
    )
boot_table = pd.concat(boot_frames, ignore_index=True)
BOOT_PATH = DATA / "forecasting_characterization_bootstrap.parquet"
boot_table.to_parquet(BOOT_PATH, index=False)
print(
    f"wrote {BOOT_PATH}  ({len(boot_table)} rows: 1 overall + {len(boot_year)} per-year strata x "
    f"{N_BOOT})"
)

# (b) Scalar summary JSON — every number the findings doc / T6 will quote.
summary = {
    "role": "F3_characterization_diagnostic",
    "note": "Re-describes the LOCKED Gate G4 sealed-test result. No retraining, no new pre-reg.",
    "n_test": N,
    "n_test_pos": N_POS,
    "n_test_neg": N_NEG,
    "base_rate": base_rate,
    "seed": SEED,
    "n_boot": N_BOOT,
    "auc": {
        "no_graph": auc_ng,
        "hgt": auc_hgt,
        "delta_pp": delta_pp,
        "margin_bar_pp": 5.0,
        "no_graph_ci": [ng_lo, ng_hi],
        "hgt_ci": [hgt_lo, hgt_hi],
    },
    "gap_bootstrap": {
        "point_pp": delta_pp,
        "mean_pp": gap_mean * 100.0,
        "ci_pp": [gap_lo * 100.0, gap_hi * 100.0],
        "se_pp": gap_se_boot * 100.0,
        "frac_replicates_gt_5pp": frac_clears_5pp,
    },
    "gap_delong": {
        "se_pp": gap_se_delong * 100.0,
        "per_auc_se_pp": {"hgt": se_hgt_dl * 100.0, "no_graph": se_ng_dl * 100.0},
        "model_score_corr": corr_dl,
    },
    "power_mde": {
        "alpha": ALPHA,
        "true_gap_bar_pp": TRUE_GAP * 100.0,
        "power_5pp_paired": power_paired,
        "power_5pp_paired_delong": power_paired_dl,
        "mde_80pct_paired_pp": mde_paired * 100.0,
        "hanley_mcneil_se_pp": {"hgt": se_hgt_hm * 100.0, "no_graph": se_ng_hm * 100.0},
        "power_5pp_independent": power_indep,
        "mde_80pct_independent_pp": mde_indep * 100.0,
        "se_independent_pp": se_indep_hm * 100.0,
    },
    "brier_lead": {
        "pvalue": float(pb["pvalue"]),
        "statistic": float(pb["statistic"]),
        "n_pairs": int(pb["n_pairs"]),
        "median_diff": float(pb["median_diff"]),
        "direction": pb["direction"],
        "recomputed_pvalue": recomputed_p,
        "hgt_worse_units": n_hgt_worse,
        "hgt_better_units": n_hgt_better,
    },
    "calibration": {
        "hgt_units_in_0p4_0p8": hgt_mid_n,
        "hgt_obs_freq_in_0p4_0p8": hgt_mid_obs,
        "no_graph_units_in_0p0_0p1": ng_low_n,
        "no_graph_obs_freq_in_0p0_0p1": ng_low_obs,
        "hgt_share_mape": float(smape["hgt"]),
        "no_graph_share_mape": float(smape["no_graph"]),
        "hgt_share_mape_is_worst_of_5": bool(worst_smape_model == "hgt"),
    },
    "winners_curse": {
        "n_trials_configured_pr2": pr2_configured_trials,
        "n_trials_ran": n_trials_total,
        "n_trials_complete": n_complete,
        "n_trials_pruned": n_pruned,
        "seeds": seeds,
        "winner_trial": winner_trial,
        "winner_val_auc": winner_val,
        "runnerup_trial": runnerup_trial,
        "runnerup_val_auc": runnerup_val,
        "winner_margin_pp": winner_margin_pp,
        "no_graph_val_auc": ng_val,
        "no_graph_test_auc": ng_test,
        "no_graph_gain_pp": ng_gain_pp,
        "hgt_val_auc": hgt_val,
        "hgt_test_auc": hgt_test,
        "hgt_gain_pp": hgt_gain_pp,
        "val_gap_pp": val_gap_pp,
        "test_gap_pp": test_gap_pp,
    },
    "per_year": per_year.to_dict(orient="records"),
}
JSON_PATH = DATA / "forecasting_characterization.json"
JSON_PATH.write_text(json.dumps(summary, indent=2, sort_keys=True))
print(f"wrote {JSON_PATH}")

# Provenance sidecars (record_run) for both new artifacts.
_inputs = {
    "per_unit": DATA / "forecasting_test_per_unit.parquet",
    "metrics": DATA / "forecasting_test_metrics.parquet",
    "calibration": DATA / "forecasting_test_calibration.parquet",
    "sweep": DATA / "forecasting_sweep.parquet",
    "baselines": DATA / "forecasting_baselines.parquet",
    "wilcoxon": DATA / "forecasting_test_wilcoxon.json",
}
_cfg = {
    "session": "V1-S16-phase1B",
    "role": "F3_characterization",
    "seed": SEED,
    "n_boot": N_BOOT,
    "reads_only": True,
    "retrained": False,
    "new_prereg": False,
    "source_per_unit_git_sha": (per_unit_sidecar or {}).get("git_sha"),
    "source_sweep_git_sha": (sweep_sidecar or {}).get("git_sha"),
}
print("sidecar:", record_run(BOOT_PATH, _inputs, {**_cfg, "artifact": "bootstrap"}))
print("sidecar:", record_run(JSON_PATH, _inputs, {**_cfg, "artifact": "summary"}))

wrote /Users/samersalman/Desktop/SciField/data/v1/forecasting_characterization_bootstrap.parquet  (30000 rows: 1 overall + 2 per-year strata x 10000)
wrote /Users/samersalman/Desktop/SciField/data/v1/forecasting_characterization.json
sidecar: /Users/samersalman/Desktop/SciField/data/v1/forecasting_characterization_bootstrap.parquet.run.json
sidecar: /Users/samersalman/Desktop/SciField/data/v1/forecasting_characterization.json.run.json


## 8. Figure — `docs/figures/F3_characterization.png`

Three panels matching the house style: (A) the bootstrap gap distribution with its 95% CI, the
+5pp bar, and the ~25% power annotation; (B) per-origin-year test AUC with bootstrap CIs (showing
how wide they are at 6-7 positives); (C) the calibration foregrounding — predicted-probability mass
vs realized emergence for HGT vs no_graph, which is the well-powered Brier story.

In [11]:
C_NG = "#4285f4"  # graph-free baseline (blue)
C_HGT = "#ea4335"  # HGT graph model (red)
C_BAR = "#222222"

fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.7))

# --- Panel A: bootstrap gap distribution + CI + +5pp bar + power ---
axA = axes[0]
g = boot_gap[np.isfinite(boot_gap)] * 100.0
axA.hist(g, bins=60, color="#9aa0a6", edgecolor="white", linewidth=0.3, zorder=2)
axA.axvline(delta_pp, color=C_HGT, lw=2.0, zorder=4, label=f"point estimate {delta_pp:+.2f}pp")
axA.axvline(gap_lo * 100, color="#444444", ls="--", lw=1.3, zorder=4)
axA.axvline(
    gap_hi * 100,
    color="#444444",
    ls="--",
    lw=1.3,
    zorder=4,
    label=f"95% CI [{gap_lo * 100:+.1f}, {gap_hi * 100:+.1f}]pp",
)
axA.axvline(5.0, color=C_BAR, ls=":", lw=1.8, zorder=5, label="+5pp pre-registered bar")
axA.axvline(0.0, color="#bbbbbb", lw=1.0, zorder=1)
axA.set_xlabel("AUC gap  (HGT - no_graph), percentage points")
axA.set_ylabel("bootstrap replicates")
axA.set_title(
    f"A. Paired AUC gap is underpowered\n{N_BOOT} resamples · power(+5pp)={power_paired:.0%} · "
    f"MDE@80%={mde_paired * 100:.0f}pp",
    fontsize=10,
)
axA.legend(fontsize=7.4, loc="upper left", framealpha=0.9)
axA.grid(axis="y", ls=":", c="#e6e6e6", zorder=0)

# --- Panel B: per-year AUC with bootstrap CIs ---
axB = axes[1]
yrs = per_year.origin_year.tolist()
xpos = np.arange(len(yrs))
off = 0.16
for i, (_, r) in enumerate(per_year.iterrows()):
    axB.errorbar(
        xpos[i] - off,
        r.auc_no_graph,
        yerr=[[r.auc_no_graph - r.auc_no_graph_lo], [r.auc_no_graph_hi - r.auc_no_graph]],
        fmt="o",
        color=C_NG,
        capsize=4,
        ms=7,
        lw=1.6,
        label="no_graph" if i == 0 else None,
    )
    axB.errorbar(
        xpos[i] + off,
        r.auc_hgt,
        yerr=[[r.auc_hgt - r.auc_hgt_lo], [r.auc_hgt_hi - r.auc_hgt]],
        fmt="s",
        color=C_HGT,
        capsize=4,
        ms=7,
        lw=1.6,
        label="hgt" if i == 0 else None,
    )
axB.axhline(0.5, ls=":", c="#bbbbbb", lw=1.0)
axB.set_xticks(xpos)
axB.set_xticklabels(
    [
        f"{int(y)}\n(n={int(n)}, {int(p)} pos)"
        for y, n, p in zip(per_year.origin_year, per_year.n, per_year.n_pos, strict=False)
    ]
)
axB.set_ylabel("emergence AUC (sealed test)")
axB.set_ylim(0.30, 1.02)
axB.set_title(
    "B. Per-origin-year AUC: huge CIs (6-7 pos/yr)\nyear swing is small-sample noise, not signal",
    fontsize=10,
)
axB.legend(fontsize=8, loc="lower left", framealpha=0.9)
axB.grid(axis="y", ls=":", c="#e6e6e6")

# --- Panel C: calibration foregrounding (predicted mass vs realized emergence) ---
axC = axes[2]
centers_h = (cal_hgt.bin_lo.values + cal_hgt.bin_hi.values) / 2
centers_n = (cal_ng.bin_lo.values + cal_ng.bin_hi.values) / 2
w = 0.045
axC.bar(
    centers_n - w / 2,
    cal_ng["n"].values,
    width=w,
    color=C_NG,
    alpha=0.85,
    label="no_graph: # units",
    zorder=2,
)
axC.bar(
    centers_h + w / 2,
    cal_hgt["n"].values,
    width=w,
    color=C_HGT,
    alpha=0.85,
    label="hgt: # units",
    zorder=2,
)
axC.axvspan(0.4, 0.8, color="#ea4335", alpha=0.07, zorder=0)
axC.axhline(0, color="#bbbbbb", lw=0.8)
axC.set_xlabel("predicted emergence probability (bin)")
axC.set_ylabel("# test units in bin")
axC.set_xlim(-0.02, 1.02)
axC.set_title(
    f"C. Calibration: HGT over-predicts (Brier p={float(pb['pvalue']):.0e})\n"
    f"HGT puts {hgt_mid_n}/{N} units in [0.4,0.8] where emergence~={hgt_mid_obs:.0%}; "
    f"no_graph stays near base rate {base_rate:.0%}",
    fontsize=9.4,
)
# overlay realized emergence frequency on a twin axis
axC2 = axC.twinx()
axC2.plot(
    centers_h,
    cal_hgt.observed_freq.values,
    "o-",
    color="#a61c00",
    ms=4,
    lw=1.2,
    label="hgt: observed freq",
    zorder=4,
)
axC2.plot(
    centers_n,
    cal_ng.observed_freq.values,
    "o-",
    color="#1a47a3",
    ms=4,
    lw=1.2,
    label="no_graph: observed freq",
    zorder=4,
)
axC2.axhline(base_rate, ls="--", c="#888888", lw=1.0, zorder=3)
axC2.set_ylabel("observed emergence frequency", fontsize=9)
axC2.set_ylim(-0.02, 1.02)
h1, l1 = axC.get_legend_handles_labels()
h2, l2 = axC2.get_legend_handles_labels()
axC.legend(h1 + h2, l1 + l2, fontsize=6.8, loc="upper center", framealpha=0.9)

fig.suptitle(
    "F3 characterization — the calibration deficit (Brier, all 138 units) is the well-powered "
    "evidence; the AUC gap is underpowered at 13 positives",
    fontsize=11,
    y=1.04,
)
fig.text(
    0.5,
    -0.04,
    f"Lead: paired per-topic Brier Wilcoxon p={float(pb['pvalue']):.2e} favors the graph-free "
    f"baseline (HGT worse-calibrated on {n_hgt_worse}/138 units). AUC gap {delta_pp:+.2f}pp, "
    f"95% CI [{gap_lo * 100:+.1f},{gap_hi * 100:+.1f}]pp, ~{power_paired:.0%} power for +5pp. "
    f"val->test flip = winner's-curse (best-of-{n_complete} trials, +{winner_margin_pp:.2f}pp over "
    "runner-up; baseline gained more, HGT did not collapse).",
    ha="center",
    fontsize=7.6,
    style="italic",
    color="#444444",
)
fig.tight_layout()
FIG_PATH = FIGURES_DIR / "F3_characterization.png"
fig.savefig(FIG_PATH, dpi=DPI, bbox_inches="tight")
plt.close(fig)
sz = FIG_PATH.stat().st_size
print(f"wrote {FIG_PATH}  ({sz / 1024:.1f} KB, dpi={DPI})")
assert sz < 1_000_000, f"figure too large: {sz} bytes"
print("size assertion PASS — under 1 MB")

# Figure provenance sidecar (house convention).
print(
    "sidecar:",
    record_run(
        FIG_PATH,
        {
            "per_unit": DATA / "forecasting_test_per_unit.parquet",
            "calibration": DATA / "forecasting_test_calibration.parquet",
            "wilcoxon": DATA / "forecasting_test_wilcoxon.json",
            "sweep": DATA / "forecasting_sweep.parquet",
        },
        {
            "figure": "F3_characterization",
            "session": "V1-S16-phase1B",
            "dpi": DPI,
            "gap_ci_pp": [gap_lo * 100, gap_hi * 100],
            "power_5pp": power_paired,
            "mde_80_pp": mde_paired * 100,
            "brier_pvalue": float(pb["pvalue"]),
            "winner_margin_pp": winner_margin_pp,
        },
    ),
)

wrote /Users/samersalman/Desktop/SciField/docs/figures/F3_characterization.png  (141.1 KB, dpi=120)
size assertion PASS — under 1 MB
sidecar: /Users/samersalman/Desktop/SciField/docs/figures/F3_characterization.png.run.json


## 9. Characterization summary (for the findings doc + T6 handoff)

All numbers below are computed above and written to `data/v1/forecasting_characterization.json`.

In [12]:
print("=" * 78)
print("F3 CHARACTERIZATION — KEY NUMBERS (faithful re-description of the LOCKED null)")
print("=" * 78)
print(f"Test set: n={N}, {N_POS} positives ({base_rate:.1%}); 2021: 67/7pos, 2022: 71/6pos.")
print()
print("[LEAD] Paired per-topic Brier Wilcoxon (all 138 units):")
print(
    f"   p = {float(pb['pvalue']):.3e}  statistic = {float(pb['statistic']):.1f}  "
    f"median_diff = {float(pb['median_diff']):+.4f}  direction = {pb['direction'].upper()}"
)
print(
    f"   HGT worse-calibrated on {n_hgt_worse}/138 units; HGT share-MAPE {float(smape['hgt']):.4f} "
    f"(worst of 5) vs no_graph {float(smape['no_graph']):.4f}."
)
print(
    f"   Mechanism: HGT puts {hgt_mid_n}/{N} units in pred-prob [0.4,0.8] where emergence"
    f"={hgt_mid_obs:.1%}; no_graph keeps {ng_low_n}/{N} in [0.0,0.1] (obs {ng_low_obs:.1%}"
    f" ~= base rate)."
)
print()
print("[UNDERPOWERED AUC ARM]")
print(
    f"   no_graph AUC {auc_ng:.4f} [{ng_lo:.4f},{ng_hi:.4f}]; hgt {auc_hgt:.4f} "
    f"[{hgt_lo:.4f},{hgt_hi:.4f}]."
)
print(
    f"   gap (hgt-no_graph) = {delta_pp:+.2f}pp; 95% bootstrap CI = "
    f"[{gap_lo * 100:+.2f}, {gap_hi * 100:+.2f}]pp (SE {gap_se_boot * 100:.2f}pp)."
)
print(f"   DeLong gap SE = {gap_se_delong * 100:.2f}pp (matches bootstrap); rho={corr_dl:.2f}.")
print(f"   power for +5pp bar = {power_paired:.1%}; MDE@80% = {mde_paired * 100:.2f}pp.")
print()
print("[WINNER'S CURSE / val->test]")
print(
    f"   sweep: {n_trials_total} ran, {n_complete} complete, {n_pruned} pruned (PR2 configured 40)."
)
print(
    f"   winner (trial {winner_trial}) val {winner_val:.4f} vs runner-up (trial {runnerup_trial}) "
    f"{runnerup_val:.4f} = +{winner_margin_pp:.2f}pp."
)
print(
    f"   val->test: no_graph +{ng_gain_pp:.1f}pp (0.{int(round(ng_val*1000)):03d}->"
    f"0.{int(round(ng_test*1000)):03d}); hgt +{hgt_gain_pp:.1f}pp (0.{int(round(hgt_val*1000)):03d}"
    f"->0.{int(round(hgt_test*1000)):03d})."
)
print(
    "   => baseline gained MORE; HGT did NOT collapse. "
    "Winner's-curse + small-sample, not regime shift."
)
print()
print("VERDICT: TRUE null — graph adds no calibration value. Characterization makes the null")
print("         rigorous (well-powered Brier lead) and explains the AUC flip (underpowered +")
print("         winner's-curse), without rescuing the graph model.")
print()
print("Artifacts written:")
for p in [BOOT_PATH, JSON_PATH, FIG_PATH]:
    print(f"   {p}  (exists={p.exists()})  + .run.json sidecar")

F3 CHARACTERIZATION — KEY NUMBERS (faithful re-description of the LOCKED null)
Test set: n=138, 13 positives (9.4%); 2021: 67/7pos, 2022: 71/6pos.

[LEAD] Paired per-topic Brier Wilcoxon (all 138 units):
   p = 2.629e-11  statistic = 1659.0  median_diff = +0.2330  direction = FAVORS_BASELINE
   HGT worse-calibrated on 125/138 units; HGT share-MAPE 0.5729 (worst of 5) vs no_graph 0.4610.
   Mechanism: HGT puts 104/138 units in pred-prob [0.4,0.8] where emergence=12.5%; no_graph keeps 129/138 in [0.0,0.1] (obs 9.3% ~= base rate).

[UNDERPOWERED AUC ARM]
   no_graph AUC 0.8043 [0.7200,0.8794]; hgt 0.7809 [0.6886,0.8646].
   gap (hgt-no_graph) = -2.34pp; 95% bootstrap CI = [-10.09, +5.60]pp (SE 3.98pp).
   DeLong gap SE = 3.98pp (matches bootstrap); rho=0.58.
   power for +5pp bar = 24.2%; MDE@80% = 11.15pp.

[WINNER'S CURSE / val->test]
   sweep: 20 ran, 5 complete, 15 pruned (PR2 configured 40).
   winner (trial 1) val 0.7367 vs runner-up (trial 4) 0.7290 = +0.77pp.
   val->test: no_grap